# LLM Zoomcamp 2026 - Homework 2: Vector Search

This notebook solves the six questions from Module 2 using the course's ONNX embedder, the lesson pages pinned to commit `8c1834d`, and `minsearch`.

The workflow follows the official homework: embed the query, compare normalized vectors with a dot product, chunk and index the lessons, compare vector and keyword retrieval, and combine both rankings with Reciprocal Rank Fusion (RRF).

## Environment

In [ ]:
# %pip install -q onnxruntime tokenizers numpy tqdm minsearch gitsource huggingface-hub


In [ ]:
import subprocess
import sys
import urllib.request
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

COURSE_COMMIT = "8c1834d"
MODEL_DIR = Path("models/Xenova/all-MiniLM-L6-v2")


### Download the course helper files and ONNX model

`urllib.request` is used instead of `wget`, so this cell works on Windows as well. The model is downloaded only when it is not already present locally.


In [ ]:
base_url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"

for filename in ["download.py", "embedder.py"]:
    if not Path(filename).exists():
        urllib.request.urlretrieve(f"{base_url}/{filename}", filename)
        print(f"Downloaded {filename}")
    else:
        print(f"Using existing {filename}")

if not (MODEL_DIR / "model.onnx").exists():
    subprocess.run([sys.executable, "download.py"], check=True)
else:
    print(f"Using existing model in {MODEL_DIR}")


In [ ]:
from embedder import Embedder

embedder = Embedder(MODEL_DIR)
print("Embedder ready")


## Q1 - Embedding a query

The ONNX version of `all-MiniLM-L6-v2` returns a normalized vector with 384 components. The question asks for its first value.


In [ ]:
query_q1 = "How does approximate nearest neighbor search work?"
v = embedder.encode(query_q1)

q1_value = float(v[0])

print("Vector shape:", v.shape)
print(f"First value: {q1_value:.6f}")
print(f"Closest homework option: {min([-0.31, -0.02, 0.12, 0.44], key=lambda x: abs(x - q1_value)):.2f}")


## Load the lesson pages

The reader is pinned to the commit specified in the homework. This keeps the corpus fixed at 72 lesson pages even if the main branch changes later.


In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COURSE_COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print("Number of lesson pages:", len(documents))
print("Document fields:", documents[0].keys())
assert len(documents) == 72


## Q2 - Cosine similarity

Since the embedder normalizes its output, the dot product is equal to cosine similarity. I compare the Q1 vector with the full content of the SQLite vector-search lesson.


In [ ]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_document = next(doc for doc in documents if doc["filename"] == target_filename)
target_vector = embedder.encode(target_document["content"])

q2_similarity = float(v.dot(target_vector))

print(f"Cosine similarity: {q2_similarity:.6f}")
print(f"Closest homework option: {min([0.07, 0.37, 0.68, 0.92], key=lambda x: abs(x - q2_similarity)):.2f}")


## Q3 - Chunking and vector search by hand

Full pages may cover several topics. I split them into overlapping chunks using the exact parameters from the homework and embed the chunk contents in batches.


In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)

print("Number of chunks:", len(chunks))
print("Chunk fields:", chunks[0].keys())


In [ ]:
batch_size = 50
chunk_vectors = []

for start in tqdm(range(0, len(chunks), batch_size)):
    texts = [chunk["content"] for chunk in chunks[start:start + batch_size]]
    chunk_vectors.extend(embedder.encode_batch(texts))

X = np.asarray(chunk_vectors)
print("Embedding matrix shape:", X.shape)


In [ ]:
scores_q3 = X.dot(v)
best_idx_q3 = int(np.argmax(scores_q3))
q3_filename = chunks[best_idx_q3]["filename"]

print("Highest score:", float(scores_q3[best_idx_q3]))
print("Q3 filename:", q3_filename)


## Q4 - Vector search with minsearch

`VectorSearch` performs the same dot-product ranking while providing a reusable search interface.


In [ ]:
vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

query_q4 = "What metric do we use to evaluate a search engine?"
vector_q4 = embedder.encode(query_q4)
results_q4 = vector_index.search(vector_q4, num_results=5)
q4_filename = results_q4[0]["filename"]

for rank, result in enumerate(results_q4, start=1):
    print(rank, result["filename"], "start=", result["start"])

print()
print("Q4 filename:", q4_filename)


## Q5 - Text search versus vector search

Both indexes use the same chunks. I retrieve the top five results for each method and compare their filenames.


In [ ]:
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
text_index.fit(chunks)

query_q5 = "How do I store vectors in PostgreSQL?"

vector_results_q5 = vector_index.search(
    embedder.encode(query_q5),
    num_results=5,
)
text_results_q5 = text_index.search(query_q5, num_results=5)

print("Vector results:")
for rank, result in enumerate(vector_results_q5, start=1):
    print(rank, result["filename"], "start=", result["start"])

print()
print("Text results:")
for rank, result in enumerate(text_results_q5, start=1):
    print(rank, result["filename"], "start=", result["start"])


In [ ]:
vector_filenames_q5 = {result["filename"] for result in vector_results_q5}
text_filenames_q5 = {result["filename"] for result in text_results_q5}
vector_only_q5 = sorted(vector_filenames_q5 - text_filenames_q5)

q5_options = [
    "02-vector-search/lessons/01-intro.md",
    "02-vector-search/lessons/02-embeddings.md",
    "02-vector-search/lessons/08-pgvector.md",
    "03-orchestration/lessons/05-rag.md",
]
q5_matches = [filename for filename in q5_options if filename in vector_only_q5]
q5_filename = q5_matches[0] if len(q5_matches) == 1 else q5_matches

print("Files found by vector search but not text search:", vector_only_q5)
print("Q5 homework option:", q5_filename)


## Q6 - Hybrid search with RRF

Reciprocal Rank Fusion combines rankings without comparing their raw scores. A chunk receives a contribution of `1 / (k + rank)` from every list in which it appears.


In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
            docs[key] = doc

    ranked_keys = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked_keys[:num_results]]


In [ ]:
query_q6 = "How do I give the model access to tools?"

vector_results_q6 = vector_index.search(
    embedder.encode(query_q6),
    num_results=5,
)
text_results_q6 = text_index.search(query_q6, num_results=5)
hybrid_results_q6 = rrf([vector_results_q6, text_results_q6])

q6_filename = hybrid_results_q6[0]["filename"]

print("Vector ranking:")
for rank, result in enumerate(vector_results_q6):
    print(rank, result["filename"], "start=", result["start"])

print()
print("Text ranking:")
for rank, result in enumerate(text_results_q6):
    print(rank, result["filename"], "start=", result["start"])

print()
print("RRF ranking:")
for rank, result in enumerate(hybrid_results_q6):
    print(rank, result["filename"], "start=", result["start"])

print()
print("Q6 filename:", q6_filename)


## Final answers

The numeric questions are matched to the closest option, as requested in the homework instructions. The remaining answers are taken directly from the computed rankings.


In [ ]:
q1_answer = min([-0.31, -0.02, 0.12, 0.44], key=lambda x: abs(x - q1_value))
q2_answer = min([0.07, 0.37, 0.68, 0.92], key=lambda x: abs(x - q2_similarity))

print("Q1 (first embedding value): ", q1_answer)
print("Q2 (cosine similarity):     ", q2_answer)
print("Q3 (best chunk filename):   ", q3_filename)
print("Q4 (first vector result):   ", q4_filename)
print("Q5 (vector-only filename):  ", q5_filename)
print("Q6 (first RRF result):      ", q6_filename)
